In [1]:
try:
    import boto3
except ImportError:
    import sys
    !{sys.executable} -m pip install -q boto3                                                
    import boto3

In [4]:
import botocore
import sagemaker
import sys
from time import gmtime, strftime
import logging
import sagemaker
from sagemaker.core.local import LocalSession

logging.getLogger("sagemaker.config").setLevel(logging.WARNING)

# Initialize the local session
sagemaker_session = LocalSession()
sagemaker_session.config = {'local': {'local_code': True}}

# PASTE YOUR ROLE ARN HERE
role = 'arn:aws:iam::493644444178:role/SageMakerLocalExecutionRole'
print("Default Sagemaker Studio IAM role: " + role)

# Test: Try to get the default S3 bucket (verifies AWS connectivity)
try:
    bucket = sagemaker_session.default_bucket()
    print(f"Success! Connected to bucket: {bucket}")
except Exception as e:
    print(f"Error: {e}")

# S3 bucket name only (no path)
bucket = "sagemaker-us-west-2-493644444178"
lab_prefix = "anomaly-detection-lab-083609365668-fe9d93f0"

print("default-bucket: " + bucket)

prefix = f"{lab_prefix}/sagemaker/rcf-benchmarks"

# S3 bucket where the original data is downloaded and stored.
downloaded_data_bucket = bucket
downloaded_data_prefix = lab_prefix
data_filename = "nyc_taxi.csv"

def check_bucket_permission(bucket):
    permission = False
    try:
        boto3.Session().client("s3").head_bucket(Bucket=bucket)
    except botocore.exceptions.ParamValidationError as e:
        print(
            "Hey! You either forgot to specify your S3 bucket"
            " or you gave your bucket an invalid name!"
        )
    except botocore.exceptions.ClientError as e:
        if e.response["Error"]["Code"] == "403":
            print(f"Hey! You don't have permission to access the bucket, {bucket}.")
        elif e.response["Error"]["Code"] == "404":
            print(f"Hey! Your bucket, {bucket}, doesn't exist!")
        else:
            raise
    else:
        permission = True
    return permission

if check_bucket_permission(bucket):
    print(f"Training input/output will be stored in: s3://{bucket}/{prefix}")
if check_bucket_permission(downloaded_data_bucket):
    print(f"Downloaded training data will be read from s3://{downloaded_data_bucket}/{downloaded_data_prefix}")

[03/18/26 11:56:49] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=843520;file:///home/cjen/mySageMaker/.venv/lib/python3.13/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=209711;file:///home/cjen/mySageMaker/.venv/lib/python3.13/site-packages/botocore/credentials.py#1392\1392]8;;\

Default Sagemaker Studio IAM role: arn:aws:iam::493644444178:role/SageMakerLocalExecutionRole
Success! Connected to bucket: sagemaker-us-west-2-493644444178
default-bucket: sagemaker-us-west-2-493644444178


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=698818;file:///home/cjen/mySageMaker/.venv/lib/python3.13/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=727219;file:///home/cjen/mySageMaker/.venv/lib/python3.13/site-packages/botocore/credentials.py#1392\1392]8;;\

Training input/output will be stored in: s3://sagemaker-us-west-2-493644444178/anomaly-detection-lab-083609365668-fe9d93f0/sagemaker/rcf-benchmarks


[03/18/26 11:56:50] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=586106;file:///home/cjen/mySageMaker/.venv/lib/python3.13/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=850645;file:///home/cjen/mySageMaker/.venv/lib/python3.13/site-packages/botocore/credentials.py#1392\1392]8;;\

Downloaded training data will be read from s3://sagemaker-us-west-2-493644444178/anomaly-detection-lab-083609365668-fe9d93f0


## Step 1: Load and Explore the Data\n\nDownload pedestrian count data from S3 and visualize it.

In [5]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["figure.dpi"] = 100

# Download data from S3
s3 = boto3.client("s3")
s3.download_file(
    downloaded_data_bucket,
    f"{downloaded_data_prefix}/{data_filename}",
    data_filename,
)

# Load into DataFrame
data = pd.read_csv(data_filename, delimiter=",")
print(data.shape)
data.head()

(10320, 2)


,timestamp,value
0,2014-07-01 00:00:00,10844
1,2014-07-01 00:30:00,8127
2,2014-07-01 01:00:00,6210
3,2014-07-01 01:30:00,4656
4,2014-07-01 02:00:00,3820


In [ ]:
# Plot raw pedestrian counts
pedestrian_counts = data.iloc[:, 0].values

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(pedestrian_counts, color="steelblue", linewidth=0.8)
ax.set_title("Pedestrian Counts Over Time")
ax.set_xlabel("Time Step")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Total samples: {len(pedestrian_counts)}")
print(f"Min: {pedestrian_counts.min()}, Max: {pedestrian_counts.max()}, Mean: {pedestrian_counts.mean():.2f}")

## Step 2: Prepare and Upload Training Data\n\nRCF expects data in RecordIO-protobuf format. We use the SageMaker SDK helper to convert and upload.

In [ ]:
from sagemaker import RandomCutForest
from sagemaker.amazon.common import RecordSerializer

# Reshape to (n_samples, n_features)
rcf_data = pedestrian_counts.reshape(-1, 1).astype("float32")

# Upload training data in RecordIO-protobuf format
training_data_channel = sagemaker.Session().upload_data(
    path=data_filename,
    bucket=bucket,
    key_prefix=f"{prefix}/train",
)
print(f"Training data uploaded to: {training_data_channel}")

## Step 3: Train the RCF Model\n\nTrain SageMaker's built-in Random Cut Forest algorithm.

In [ ]:
rcf = RandomCutForest(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    data_location=f"s3://{bucket}/{prefix}/train/",
    output_path=f"s3://{bucket}/{prefix}/output",
    num_samples_per_tree=512,
    num_trees=50,
    sagemaker_session=sess,
)

rcf.fit(rcf.record_set(rcf_data))

## Step 4: Deploy Endpoint and Run Inference

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

rcf_predictor = rcf.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
)
print(f"Endpoint name: {rcf_predictor.endpoint_name}")

In [ ]:
# Run inference in batches to avoid payload size limits
def get_anomaly_scores(predictor, data, batch_size=500):
    scores = []
    for i in range(0, len(data), batch_size):
        batch = data[i : i + batch_size]
        result = predictor.predict(batch)
        scores += [r["score"] for r in result["scores"]]
    return np.array(scores)

anomaly_scores = get_anomaly_scores(rcf_predictor, rcf_data)
print(f"Computed {len(anomaly_scores)} anomaly scores")
print(f"Score stats — min: {anomaly_scores.min():.2f}, max: {anomaly_scores.max():.2f}, mean: {anomaly_scores.mean():.2f}")

## Step 5: Visualize Anomalies\n\nFlag points whose score exceeds 3 standard deviations above the mean.

In [ ]:
score_mean = anomaly_scores.mean()
score_std = anomaly_scores.std()
anomaly_threshold = score_mean + 3 * score_std

anomalies = np.where(anomaly_scores > anomaly_threshold)[0]
print(f"Threshold: {anomaly_threshold:.2f}")
print(f"Anomalies detected: {len(anomalies)} at indices: {anomalies}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Raw counts with anomalies highlighted
ax1.plot(pedestrian_counts, color="steelblue", linewidth=0.8, label="Pedestrian Count")
ax1.scatter(anomalies, pedestrian_counts[anomalies], color="red", zorder=5, label="Anomaly")
ax1.set_title("Pedestrian Counts with Detected Anomalies")
ax1.set_ylabel("Count")
ax1.legend()

# Anomaly scores
ax2.plot(anomaly_scores, color="darkorange", linewidth=0.8, label="Anomaly Score")
ax2.axhline(anomaly_threshold, color="red", linestyle="--", linewidth=1, label=f"Threshold ({anomaly_threshold:.2f})")
ax2.scatter(anomalies, anomaly_scores[anomalies], color="red", zorder=5)
ax2.set_title("RCF Anomaly Scores")
ax2.set_xlabel("Time Step")
ax2.set_ylabel("Score")
ax2.legend()

plt.tight_layout()
plt.show()

## Step 6: Cleanup\n\nDelete the endpoint to avoid ongoing charges.

In [ ]:
rcf_predictor.delete_endpoint()
print("Endpoint deleted.")